In [1]:
!pip install requests beautifulsoup4 pandas plotly streamlit -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 33.3 MB/s eta 0:00:00
Selecting previously unselected package cloudflared.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.3.0) ...
Setting up cloudflared (2026.3.0) ...
Processing triggers for man-db (2.10.2-1) ...


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
from datetime import datetime

In [4]:
from google.colab import userdata

sender_email = userdata.get("sender_email")
app_password = userdata.get("app_password")
recipient_email = userdata.get("recipient_email")
api_key = userdata.get("api_key")

print("Secrets loaded:")
print("Sender:", "OK" if sender_email else "Missing")
print("Password:", "OK" if app_password else "Missing")
print("Recipient:", "OK" if recipient_email else "Missing")
print("API Key:", "OK" if api_key else "Missing")

Secrets loaded:
Sender: OK
Password: OK
Recipient: OK
API Key: OK


In [5]:
def scan_website(url):

    vulnerabilities = []

    try:

        r = requests.get(url,timeout=5)
        headers = r.headers
        soup = BeautifulSoup(r.text,"html.parser")

        # 1 HTTPS check
        if not url.startswith("https"):
            vulnerabilities.append(("No HTTPS","High"))

        # 2 Security headers
        if "X-Frame-Options" not in headers:
            vulnerabilities.append(("Missing X-Frame-Options","Medium"))

        if "Content-Security-Policy" not in headers:
            vulnerabilities.append(("Missing CSP Header","High"))

        if "X-Content-Type-Options" not in headers:
            vulnerabilities.append(("Missing X-Content-Type-Options","Medium"))

        # 3 Form without action
        forms = soup.find_all("form")

        for form in forms:
            if not form.get("action"):
                vulnerabilities.append(("Form without action","Low"))

        # 4 Directory listing
        if "Index of /" in r.text:
            vulnerabilities.append(("Directory Listing Enabled","High"))

        # 5 Server header exposed
        if "Server" in headers:
            vulnerabilities.append(("Server Header Exposed","Low"))

    except:
        vulnerabilities.append(("Website unreachable","Critical"))

    return vulnerabilities

In [6]:
targets = [
    "http://testphp.vulnweb.com",
    "http://testasp.vulnweb.com",
    "http://zero.webappsecurity.com"
]

results = []

for url in targets:

    print("Scanning:",url)

    vulns = scan_website(url)

    for v in vulns:

        results.append({
            "target":url,
            "issue":v[0],
            "severity":v[1],
            "timestamp":datetime.now()
        })

df = pd.DataFrame(results)

df

Scanning: http://testphp.vulnweb.com
Scanning: http://testasp.vulnweb.com
Scanning: http://zero.webappsecurity.com


,target,issue,severity,timestamp
0,http://testphp.vulnweb.com,Website unreachable,Critical,2026-03-16 12:14:19.779659
1,http://testasp.vulnweb.com,No HTTPS,High,2026-03-16 12:14:19.948089
2,http://testasp.vulnweb.com,Missing X-Frame-Options,Medium,2026-03-16 12:14:19.948096
3,http://testasp.vulnweb.com,Missing CSP Header,High,2026-03-16 12:14:19.948097
4,http://testasp.vulnweb.com,Missing X-Content-Type-Options,Medium,2026-03-16 12:14:19.948098
5,http://testasp.vulnweb.com,Server Header Exposed,Low,2026-03-16 12:14:19.948100
6,http://zero.webappsecurity.com,No HTTPS,High,2026-03-16 12:14:20.343873
7,http://zero.webappsecurity.com,Missing X-Frame-Options,Medium,2026-03-16 12:14:20.343879
8,http://zero.webappsecurity.com,Missing CSP Header,High,2026-03-16 12:14:20.343880
9,http://zero.webappsecurity.com,Missing X-Content-Type-Options,Medium,2026-03-16 12:14:20.343881


In [7]:
severity_weight = {
    "Critical":10,
    "High":7,
    "Medium":4,
    "Low":2
}

df["risk_score"] = df["severity"].map(severity_weight)

risk_summary = df.groupby("target")["risk_score"].sum().reset_index()

risk_summary

,target,risk_score
0,http://testasp.vulnweb.com,24
1,http://testphp.vulnweb.com,10
2,http://zero.webappsecurity.com,24


In [8]:
import smtplib
from email.mime.text import MIMEText

high = df[df["severity"].isin(["High","Critical"])]

if high.empty:

    print("No High/Critical vulnerabilities — no email sent")

else:

    body = "<h2>🚨 Vulnerability Alert</h2>"
    body += "<table border=1>"
    body += "<tr><th>Target</th><th>Issue</th><th>Severity</th></tr>"

    for _,row in high.iterrows():

        body += f"""
        <tr>
        <td>{row['target']}</td>
        <td>{row['issue']}</td>
        <td>{row['severity']}</td>
        </tr>
        """

    body += "</table>"
    body += "<p>Recommended: Fix headers and enforce HTTPS.</p>"
    body += "<hr><p>Automated Security Scanner</p>"

    msg = MIMEText(body,"html")

    msg["Subject"] = "🚨 High/Critical Vulnerabilities Detected"
    msg["From"] = sender_email
    msg["To"] = recipient_email

    server = smtplib.SMTP("smtp.gmail.com",587)

    server.starttls()

    server.login(sender_email,app_password)

    server.send_message(msg)

    server.quit()

    print("Email alert sent successfully")

Email alert sent successfully


In [9]:
df.to_csv("scan_results.csv",index=False)

print("scan_results.csv saved")

scan_results.csv saved


In [10]:
%%writefile dashboard.py

import streamlit as st
import pandas as pd
import plotly.express as px
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import smtplib
from email.mime.text import MIMEText

st.set_page_config(
    page_title="CyberScan Dashboard",
    page_icon="🛡️",
    layout="wide"
)

st.title("🛡 Web Vulnerability Risk Dashboard")

# -----------------------------
# SIDEBAR CONFIGURATION
# -----------------------------

st.sidebar.title("⚙ Configuration")

targets = st.sidebar.multiselect(
    "Select Target Websites",
    [
        "http://testphp.vulnweb.com",
        "http://testasp.vulnweb.com",
        "http://zero.webappsecurity.com"
    ],
    default=["http://testphp.vulnweb.com"]
)

sender_email = st.sidebar.text_input("Sender Email")
app_password = st.sidebar.text_input("App Password", type="password")
recipient_email = st.sidebar.text_input("Recipient Email")

scan_button = st.sidebar.button("🚀 Run Scan")

# -----------------------------
# SCANNER FUNCTION
# -----------------------------

def scan_website(url):

    vulnerabilities = []

    try:

        r = requests.get(url,timeout=5)
        headers = r.headers
        soup = BeautifulSoup(r.text,"html.parser")

        if not url.startswith("https"):
            vulnerabilities.append(("No HTTPS","High"))

        if "X-Frame-Options" not in headers:
            vulnerabilities.append(("Missing X-Frame-Options","Medium"))

        if "Content-Security-Policy" not in headers:
            vulnerabilities.append(("Missing CSP Header","High"))

        if "X-Content-Type-Options" not in headers:
            vulnerabilities.append(("Missing X-Content-Type-Options","Medium"))

        forms = soup.find_all("form")

        for form in forms:
            if not form.get("action"):
                vulnerabilities.append(("Form without action","Low"))

        if "Index of /" in r.text:
            vulnerabilities.append(("Directory Listing Enabled","High"))

        if "Server" in headers:
            vulnerabilities.append(("Server Header Exposed","Low"))

    except:

        vulnerabilities.append(("Website unreachable","Critical"))

    return vulnerabilities


# -----------------------------
# RUN SCAN
# -----------------------------

if scan_button:

    results = []

    for url in targets:

        vulns = scan_website(url)

        for v in vulns:

            results.append({
                "target":url,
                "issue":v[0],
                "severity":v[1],
                "timestamp":datetime.now()
            })

    df = pd.DataFrame(results)

    severity_weight = {
        "Critical":10,
        "High":7,
        "Medium":4,
        "Low":2
    }

    df["risk_score"] = df["severity"].map(severity_weight)

    st.session_state["df"] = df

    # -----------------------------
    # EMAIL ALERT
    # -----------------------------

    high = df[df["severity"].isin(["High","Critical"])]

    if not high.empty and sender_email and app_password and recipient_email:

        body = "<h2>🚨 Vulnerability Alert</h2>"
        body += "<table border=1>"
        body += "<tr><th>Target</th><th>Issue</th><th>Severity</th></tr>"

        for _,row in high.iterrows():

            body += f"""
            <tr>
            <td>{row['target']}</td>
            <td>{row['issue']}</td>
            <td>{row['severity']}</td>
            </tr>
            """

        body += "</table>"
        body += "<p>Recommended: Fix headers and enforce HTTPS.</p>"
        body += "<hr><p>Automated Security Scanner</p>"

        msg = MIMEText(body,"html")

        msg["Subject"] = "🚨 High/Critical Vulnerabilities Detected"
        msg["From"] = sender_email
        msg["To"] = recipient_email

        server = smtplib.SMTP("smtp.gmail.com",587)

        server.starttls()

        server.login(sender_email,app_password)

        server.send_message(msg)

        server.quit()

        st.sidebar.success("Email alert sent!")

# -----------------------------
# LOAD DATA
# -----------------------------

if "df" in st.session_state:

    df = st.session_state["df"]

else:

    st.info("Run a scan using the sidebar")

    st.stop()


# -----------------------------
# KPI METRICS
# -----------------------------

st.subheader("📊 Security Overview")

total = len(df)
critical = len(df[df["severity"]=="Critical"])
high = len(df[df["severity"]=="High"])
medium = len(df[df["severity"]=="Medium"])

c1,c2,c3,c4 = st.columns(4)

c1.metric("Total Vulnerabilities", total)
c2.metric("Critical", critical)
c3.metric("High", high)
c4.metric("Medium", medium)

st.divider()

# -----------------------------
# TABS
# -----------------------------

tab1,tab2,tab3 = st.tabs([
    "📋 Scan Results",
    "📈 Charts",
    "🚨 Threat Analysis"
])

with tab1:

    st.dataframe(df, use_container_width=True)

with tab2:

    fig1 = px.pie(df,names="severity",title="Severity Distribution")
    st.plotly_chart(fig1,use_container_width=True)

    fig2 = px.bar(df,x="issue",color="severity",title="Vulnerability Types")
    st.plotly_chart(fig2,use_container_width=True)

    fig3 = px.histogram(df,x="severity",title="Severity Histogram")
    st.plotly_chart(fig3,use_container_width=True)

    fig4 = px.bar(df,x="target",color="severity",title="Vulnerabilities per Website")
    st.plotly_chart(fig4,use_container_width=True)

    risk = df.groupby("target")["risk_score"].sum().reset_index()

    fig5 = px.bar(risk,x="target",y="risk_score",title="Risk Score per Target")

    st.plotly_chart(fig5,use_container_width=True)

with tab3:

    high = df[df["severity"].isin(["High","Critical"])]

    if high.empty:

        st.success("No High Risk Issues")

    else:

        st.error(f"{len(high)} High Risk Issues Detected")

        st.dataframe(high)

Writing dashboard.py


In [11]:
import subprocess, threading, time

def run_streamlit():
    subprocess.run(["streamlit","run","dashboard.py","--server.port","8501","--server.headless","true"])

threading.Thread(target=run_streamlit,daemon=True).start()

time.sleep(5)

print("Streamlit started")


Streamlit started


In [ ]:
!cloudflared tunnel --url http://localhost:8501

2026-03-16T12:15:44Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-16T12:15:44Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-16T12:15:52Z INF +--------------------------------------------------------------------------------------------+
2026-03-16T12:15:52Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-03-16T12:15:52Z INF |  https://west-amy-stage-distributions.trycloudflare.co